<a href="https://colab.research.google.com/github/ahmedlayouni2001/WiseVision/blob/main/prise_en_charge_client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [Cell 1] - Imports
import cv2, os, glob, numpy as np, pandas as pd
from collections import defaultdict, deque
from dataclasses import dataclass
print("cv2", cv2.__version__)

# %% [Cell 2] - Config + auto-find inputs
CANDIDATE_SECONDS        = 2.0
CONFIRMED_SECONDS        = 5.0
INTERACTION_RATIO_THRESH = 0.7
DIST_MULT                = 1.3
GRACE_SECONDS            = 1.5
END_SECONDS              = 1.5
SMOOTH_WINDOW            = 5
COLLEAGUE_SKIP           = 2   # must match the SKIP used in notebook 1

VIDEO_OUT  = "/kaggle/working/pec_output.mp4"
VIDEO_H264 = "/kaggle/working/pec_output_h264.mp4"
CSV_OUT    = "/kaggle/working/prise_en_charge.csv"

def _find(patterns):
    for pat in patterns:
        hits = glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
        if hits: return sorted(hits)[0]
    return None

# Prefer the ORIGINAL video (cleaner). If you only have the annotated one, it works too.
VIDEO_IN  = _find(["0509.mp4", "input.mp4", "shop_output_v4.mp4", "*.mp4"])
TRACKS_IN = _find(["tracks_export.csv", "tracks*.csv"])
assert VIDEO_IN,  "No video in /kaggle/input"
assert TRACKS_IN, "No tracks_export.csv in /kaggle/input"
print("VIDEO :", VIDEO_IN)
print("TRACKS:", TRACKS_IN)


# %% [Cell 3] - Load tracks
df = pd.read_csv(TRACKS_IN)
print(f"{len(df)} rows | frames {df.frame.min()}..{df.frame.max()} "
      f"| roles: {df.role.value_counts().to_dict()}")

frames = defaultdict(list)
for r in df.itertuples(index=False):
    frames[int(r.frame)].append({
        "first_pid": int(r.first_pid),
        "role":      r.role.upper(),
        "bbox":      np.array([r.x1, r.y1, r.x2, r.y2], dtype=float),
        "timestamp": float(r.timestamp),
    })
print(f"Loaded {len(frames)} processed frames")


# %% [Cell 4] - PEC engine (FSM + smoother + drawer)
def foot_point(b):  return np.array([(b[0]+b[2])/2.0, float(b[3])])
def bbox_height(b): return float(b[3]-b[1])
def pair_key(a,b):  return tuple(sorted((a,b)))

class FootSmoother:
    def __init__(self, w=5): self.buf = defaultdict(lambda: deque(maxlen=w))
    def update(self, tid, pt):
        self.buf[tid].append(np.asarray(pt, dtype=float))
        return np.array(self.buf[tid]).mean(axis=0)

S_NONE,S_CAND,S_CONF,S_END = 0,1,2,3
NAME = {0:"NONE",1:"CANDIDATE",2:"PRISE EN CHARGE",3:"ENDED"}

@dataclass
class PairState:
    cid:int; vid:int; state:int=S_NONE
    close:int=0; total:int=0; near:int=0; far:int=0
    start_t:float=0.0; grace:int=0
    def ratio(self): return self.close/self.total if self.total else 0.0

class PECManager:
    def __init__(self, fps):
        self.fps=fps; self.pairs={}; self.events=[]
        self.smoother = FootSmoother(SMOOTH_WINDOW)
        self.cand_thr = max(1,int(CANDIDATE_SECONDS*fps))
        self.conf_thr = max(1,int(CONFIRMED_SECONDS*fps))
        self.end_thr  = max(1,int(END_SECONDS*fps))
        self.gthr     = max(1,int(GRACE_SECONDS*fps))
        self.seller_clients = defaultdict(set)   # vendor_id -> {client_id, ...}

    def update(self, t, persons):
        clients = [p for p in persons if p["role"]=="CLIENT"]
        vendors = [p for p in persons if p["role"]=="SELLER"]
        foot = {p["first_pid"]: self.smoother.update(p["first_pid"], foot_point(p["bbox"]))
                for p in persons}
        seen, links = set(), []
        for c in clients:
            for v in vendors:
                cid,vid = c["first_pid"], v["first_pid"]
                k = pair_key(cid,vid); seen.add(k)
                ps = self.pairs.setdefault(k, PairState(cid=cid,vid=vid))
                d   = float(np.linalg.norm(foot[cid]-foot[vid]))
                thr = DIST_MULT*(bbox_height(c["bbox"])+bbox_height(v["bbox"]))/2.0
                close = d<=thr
                ps.total += 1
                if close: ps.close+=1; ps.near+=1; ps.far=0; ps.grace=self.gthr
                else:     ps.far+=1; ps.near=0; ps.grace=max(0,ps.grace-1)
                self._trans(ps, t)
                if ps.state in (S_CAND,S_CONF): links.append((c,v,ps))
        for k,ps in self.pairs.items():
            if k in seen: continue
            if ps.state in (S_CAND,S_CONF):
                ps.far+=1; ps.grace=max(0,ps.grace-1); self._trans(ps,t)
        return links

    def _trans(self, ps, t):
        if ps.state==S_NONE:
            if ps.near>=self.cand_thr:
                ps.state=S_CAND; ps.start_t = t - ps.near/self.fps
        elif ps.state==S_CAND:
            if ps.near>=self.conf_thr and ps.ratio()>INTERACTION_RATIO_THRESH:
                ps.state=S_CONF
                self.seller_clients[ps.vid].add(ps.cid)
                print(f"  [PEC START] client #{ps.cid} ← seller #{ps.vid} "
                      f"at {int(ps.start_t)//60:02d}:{int(ps.start_t)%60:02d}")
            elif ps.far>self.end_thr and ps.grace==0:
                ps.state=S_NONE; ps.close=ps.total=0
        elif ps.state==S_CONF:
            if ps.far>self.end_thr and ps.grace==0:
                self._close(ps,t); ps.state=S_END
        elif ps.state==S_END:
            ps.state=S_NONE; ps.close=ps.total=0; ps.near=ps.far=0

    def _close(self, ps, end_t):
        dur = end_t - ps.start_t
        self.events.append({
            "client_id":ps.cid, "vendor_id":ps.vid,
            "start_s":round(ps.start_t,2), "end_s":round(end_t,2),
            "duration_s":round(dur,2),
            "start_mmss":f"{int(ps.start_t)//60:02d}:{int(ps.start_t)%60:02d}",
            "end_mmss":  f"{int(end_t)//60:02d}:{int(end_t)%60:02d}",
        })
        print(f"  [PEC END]   client #{ps.cid} duration {dur:.1f}s")

    def finalize(self, t):
        for ps in self.pairs.values():
            if ps.state==S_CONF: self._close(ps,t)

def draw_links(frame, links, fps):
    for c,v,ps in links:
        cfx,cfy = map(int, foot_point(c["bbox"]))
        vfx,vfy = map(int, foot_point(v["bbox"]))
        color = (0,165,255) if ps.state==S_CAND else (0,0,255)
        cv2.line(frame,(cfx,cfy),(vfx,vfy),color,3)
        mx,my = (cfx+vfx)//2, (cfy+vfy)//2
        timer = ps.near/fps
        cv2.putText(frame,f"{NAME[ps.state]} {timer:.1f}s",(mx-100,my),
                    cv2.FONT_HERSHEY_SIMPLEX,0.8,color,2)
        if ps.state==S_CONF:
            cv2.putText(frame,"PRISE EN CHARGE",(mx-130,my-35),
                        cv2.FONT_HERSHEY_SIMPLEX,0.9,color,3)


# %% [Cell 5] - Run PEC on the video
cap = cv2.VideoCapture(VIDEO_IN)
fps_raw = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
eff_fps = fps_raw / COLLEAGUE_SKIP
print(f"Video: {W}x{H} @ {fps_raw}fps, {N} frames | effective_fps={eff_fps}")

writer = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*"mp4v"),
                        fps_raw, (W,H))
assert writer.isOpened(), "VideoWriter failed to open"

mgr = PECManager(eff_fps)
last_links, processed_idx, last_t = [], -1, 0.0

for fi in range(N):
    ok, frame = cap.read()
    if not ok: break
    if fi % COLLEAGUE_SKIP == 0:
        processed_idx += 1
        persons = frames.get(processed_idx, [])
        t = persons[0]["timestamp"] if persons else fi/fps_raw
        last_t = t
        last_links = mgr.update(t, persons)
        for p in persons:
            x1,y1,x2,y2 = map(int, p["bbox"])
            color = (0,0,255) if p["role"]=="SELLER" else (0,200,0)
            label = (f"SELLER" if p["role"]=="SELLER"
                     else f"CLIENT #{p['first_pid']}")
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,label,(x1,max(15,y1-6)),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)
    draw_links(frame, last_links, eff_fps)
    writer.write(frame)

mgr.finalize(last_t)
cap.release(); writer.release()

print("\n========== CLIENTS PER SELLER ==========")
for vid, clients in sorted(mgr.seller_clients.items()):
    print(f"  Seller #{vid}: {len(clients)} client(s)")
print("========================================")

pd.DataFrame(mgr.events).to_csv(CSV_OUT, index=False)
print(f"\n✓ Video : {VIDEO_OUT}")
print(f"✓ CSV   : {CSV_OUT}")
print(f"✓ Events: {len(mgr.events)}")


# %% [Cell 6] - Re-encode to H.264 for browser playback
import subprocess
r = subprocess.run(["ffmpeg","-y","-i",VIDEO_OUT,"-vcodec","libx264",
                    "-pix_fmt","yuv420p","-crf","23",VIDEO_H264],
                   capture_output=True, text=True)
print("ffmpeg ok" if r.returncode==0 else r.stderr[-600:])



# %% [Cell 7] - Preview + show events table
from IPython.display import HTML, display
from base64 import b64encode

events_df = pd.read_csv(CSV_OUT) if os.path.exists(CSV_OUT) else pd.DataFrame()
print("=== PRISE EN CHARGE EVENTS ===")
display(events_df)

def show(path, w=720, max_mb=40):
    if not os.path.exists(path): return HTML(f"Not found: {path}")
    sz = os.path.getsize(path)/1e6
    if sz > max_mb:
        return HTML(f"{sz:.1f}MB — too large to embed. Download from Output tab.")
    d = b64encode(open(path,"rb").read()).decode()
    return HTML(f'<video width={w} controls><source src="data:video/mp4;base64,{d}" type="video/mp4"></video>')
show(VIDEO_H264)